# Modelado — Machine Predictive Maintenance

**Destino de despliegue**: SageMaker real-time inference endpoint  
**Container**: `sagemaker-scikit-learn:1.2-1-cpu-py3`

## 1. Alcance

Se entrenan tres modelos de clasificación binaria (`failure`) con el mismo pipeline de preprocesamiento. El pipeline es un objeto `sklearn.pipeline.Pipeline` que incluye:

1. **`FeatureEngineer`** — agrega `power`, `temp_diff`, `strain` a partir de los features de entrada
2. **`ColumnTransformer`** — escala numéricas (`StandardScaler`), codifica `machine_type` (`OrdinalEncoder`)
3. **Clasificador** — intercambiable: LR, RF, HistGradientBoosting

El modelo final se serializa como `model.joblib` y se empaqueta con `inference.py` en `model.tar.gz`, el artefacto que SageMaker espera.

**Por qué `sagemaker-scikit-learn` y no `sagemaker-xgboost`**  
El container de XGBoost espera datos en formato DMatrix (CSV o LibSVM) y corre XGBoost nativo. No puede servir un `sklearn.pipeline.Pipeline`. El container de sklearn soporta pipelines arbitrarios y código de inferencia personalizado vía `inference.py`.

In [ ]:
import joblib
import tarfile
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve
)

TEMPLATE = 'plotly_white'
DATA_DIR = Path('data')
RANDOM_STATE = 42

## 2. Datos

In [ ]:
df = pd.read_csv(DATA_DIR / 'predictive_maintenance.csv', encoding='utf-8-sig')
df = df.rename(columns={
    'UDI': 'id', 'Product ID': 'product_id', 'Type': 'machine_type',
    'Air temperature [K]': 'air_temp', 'Process temperature [K]': 'process_temp',
    'Rotational speed [rpm]': 'speed', 'Torque [Nm]': 'torque',
    'Tool wear [min]': 'tool_wear', 'Target': 'target', 'Failure Type': 'failure_type'
})

# Target desde failure_type (fuente de verdad)
df['failure'] = (df['failure_type'] != 'No Failure').astype(int)

INPUT_FEATURES = ['air_temp', 'process_temp', 'speed', 'torque', 'tool_wear', 'machine_type']
X = df[INPUT_FEATURES]
y = df['failure']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Tasa de fallos — train: {y_train.mean():.4f}  |  test: {y_test.mean():.4f}")

## 3. Pipeline de preprocesamiento

`FeatureEngineer` debe definirse con el mismo nombre de clase tanto en el notebook de entrenamiento como en `inference.py`. Cuando `joblib` deserializa el modelo en SageMaker, Python necesita importar la clase desde el módulo activo — si no existe con el mismo nombre, lanza `AttributeError`. Por eso `inference.py` redefine `FeatureEngineer` exactamente.

In [ ]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=INPUT_FEATURES)
        X['power']     = X['torque'] * (X['speed'] * 2 * np.pi / 60)
        X['temp_diff'] = X['process_temp'] - X['air_temp']
        X['strain']    = X['torque'] * X['tool_wear']
        return X


NUM_COLS = ['air_temp', 'process_temp', 'speed', 'torque', 'tool_wear',
            'power', 'temp_diff', 'strain']
CAT_COLS = ['machine_type']


def build_pipeline(classifier):
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), NUM_COLS),
        ('cat', OrdinalEncoder(categories=[['H', 'M', 'L']]), CAT_COLS)
    ], remainder='drop')
    return Pipeline([
        ('eng', FeatureEngineer()),
        ('pre', preprocessor),
        ('clf', classifier)
    ])


eval_results = {}

def evaluate(name, pipeline, X_te, y_te):
    y_prob = pipeline.predict_proba(X_te)[:, 1]
    y_pred = pipeline.predict(X_te)
    print(f"\n=== {name} ===")
    print(classification_report(y_te, y_pred,
                                target_names=['No failure', 'Failure'], digits=3))
    roc = roc_auc_score(y_te, y_prob)
    ap  = average_precision_score(y_te, y_prob)
    print(f"AUC-ROC: {roc:.4f}  |  Avg Precision (PR-AUC): {ap:.4f}")
    eval_results[name] = {'auc_roc': roc, 'ap': ap, 'y_prob': y_prob}

## 4. Regresión logística — baseline

`class_weight='balanced'` pondera cada observación por `n_samples / (n_classes * n_samples_per_class)`, compensando el desbalance sin modificar los datos. Es equivalente a sobremuestrear la clase minoritaria antes de entrenar.

In [ ]:
lr_pipeline = build_pipeline(
    LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
)
lr_pipeline.fit(X_train, y_train)
evaluate('Logistic Regression', lr_pipeline, X_test, y_test)

## 5. Random Forest

In [ ]:
rf_pipeline = build_pipeline(
    RandomForestClassifier(
        n_estimators=200, class_weight='balanced',
        n_jobs=-1, random_state=RANDOM_STATE
    )
)
rf_pipeline.fit(X_train, y_train)
evaluate('Random Forest', rf_pipeline, X_test, y_test)

## 6. Gradient Boosting

`HistGradientBoostingClassifier` es el equivalente a XGBoost/LightGBM dentro de scikit-learn (histogramas para splits, soporte nativo de `NaN`, `class_weight`). Se elige sobre XGBoost porque no agrega dependencias externas y el container `sagemaker-scikit-learn` no incluye xgboost preinstalado.

In [ ]:
hgbc_pipeline = build_pipeline(
    HistGradientBoostingClassifier(
        class_weight='balanced', max_iter=300,
        learning_rate=0.05, random_state=RANDOM_STATE
    )
)
hgbc_pipeline.fit(X_train, y_train)
evaluate('HistGradientBoosting', hgbc_pipeline, X_test, y_test)

## 7. Comparación

Con desbalance severo (3.6% positivos), el AUC-ROC puede ser engañoso porque la clase negativa domina. **Avg Precision (PR-AUC)** es la métrica principal: mide la calidad de las predicciones sobre la clase positiva en todos los umbrales.

In [ ]:
# Tabla comparativa
comp_df = pd.DataFrame({
    name: {'AUC-ROC': v['auc_roc'], 'Avg Precision': v['ap']}
    for name, v in eval_results.items()
}).T.round(4)
print(comp_df)

# Curvas ROC
fig = go.Figure()
for name, v in eval_results.items():
    fpr, tpr, _ = roc_curve(y_test, v['y_prob'])
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr, mode='lines',
        name=f"{name} (AUC={v['auc_roc']:.3f})"
    ))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines',
                         line=dict(dash='dash', color='gray'), showlegend=False))
fig.update_layout(title='Curvas ROC', xaxis_title='FPR', yaxis_title='TPR',
                  template=TEMPLATE)
fig.show()

# Curvas Precision-Recall
fig2 = go.Figure()
for name, v in eval_results.items():
    prec, rec, _ = precision_recall_curve(y_test, v['y_prob'])
    fig2.add_trace(go.Scatter(
        x=rec, y=prec, mode='lines',
        name=f"{name} (AP={v['ap']:.3f})"
    ))
baseline = y_test.mean()
fig2.add_hline(y=baseline, line_dash='dash', line_color='gray',
               annotation_text=f'baseline ({baseline:.3f})')
fig2.update_layout(title='Curvas Precision-Recall',
                   xaxis_title='Recall', yaxis_title='Precision',
                   template=TEMPLATE)
fig2.show()

## 8. Calibración

El endpoint devuelve `failure_prob` como probabilidad. Si esa probabilidad no está calibrada, fijar un umbral de alerta (por ejemplo 0.3) no tiene significado estadístico — un modelo que predice 0.8 para el 5% de los casos puede tener en realidad solo 30% de positivos reales en esa región.

`CalibratedClassifierCV` con `method='isotonic'` y `cv=5` entrena internamente 5 folds y aprende una función monótona que mapea la salida del clasificador a frecuencias reales. El modelo base no se modifica.

In [ ]:
# Seleccionar el mejor modelo por Avg Precision
best_name = max(eval_results, key=lambda n: eval_results[n]['ap'])
pipeline_map = {
    'Logistic Regression': lr_pipeline,
    'Random Forest': rf_pipeline,
    'HistGradientBoosting': hgbc_pipeline
}
best_pipeline = pipeline_map[best_name]
print(f"Modelo base seleccionado: {best_name}")

calibrated = CalibratedClassifierCV(best_pipeline, method='isotonic', cv=5)
calibrated.fit(X_train, y_train)

# Curva de calibración
y_prob_base = best_pipeline.predict_proba(X_test)[:, 1]
y_prob_cal  = calibrated.predict_proba(X_test)[:, 1]

frac_base, mean_base = calibration_curve(y_test, y_prob_base, n_bins=10)
frac_cal,  mean_cal  = calibration_curve(y_test, y_prob_cal,  n_bins=10)

fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines',
                         line=dict(dash='dash', color='gray'), name='Perfecta'))
fig.add_trace(go.Scatter(x=mean_base, y=frac_base, mode='lines+markers',
                         name=f'{best_name} (sin calibrar)'))
fig.add_trace(go.Scatter(x=mean_cal, y=frac_cal, mode='lines+markers',
                         name='Calibrado (isotonic)'))
fig.update_layout(title='Curva de calibración',
                  xaxis_title='Probabilidad predicha',
                  yaxis_title='Fracción positiva real',
                  template=TEMPLATE)
fig.show()

evaluate('Calibrado', calibrated, X_test, y_test)

## 9. Importancia de features

Permutation importance mide la caída en Avg Precision cuando se permuta aleatoriamente cada feature de entrada (sobre los datos de test, después de calibración). Mide importancia sobre las features **crudas** — las que llegan al endpoint en producción.

In [ ]:
perm = permutation_importance(
    calibrated, X_test, y_test,
    n_repeats=15, random_state=RANDOM_STATE, scoring='average_precision'
)

imp_df = pd.DataFrame({
    'feature': INPUT_FEATURES,
    'importance': perm.importances_mean,
    'std': perm.importances_std
}).sort_values('importance', ascending=False)

fig = px.bar(
    imp_df, x='feature', y='importance', error_y='std',
    title='Importancia de features — permutation importance (Avg Precision)',
    labels={'feature': 'Feature', 'importance': 'Caída en AP'},
    template=TEMPLATE
)
fig.show()
print(imp_df.to_string(index=False))

## 10. Clasificación multiclase — tipo de fallo

El mismo pipeline con RF entrena sobre `failure_type` (6 clases incluyendo 'No Failure'). En producción, este modelo devuelve tanto la detección de fallo como su causa probable en un solo llamado al endpoint. Se evalúa con macro-F1 para tratar todas las clases por igual.

In [ ]:
# Usar los mismos índices de split del modelo binario
y_multi_train = df.loc[X_train.index, 'failure_type']
y_multi_test  = df.loc[X_test.index,  'failure_type']

rf_multi = build_pipeline(
    RandomForestClassifier(
        n_estimators=200, class_weight='balanced',
        n_jobs=-1, random_state=RANDOM_STATE
    )
)
rf_multi.fit(X_train, y_multi_train)
print(classification_report(y_multi_test, rf_multi.predict(X_test), digits=3))

## 11. Serialización para SageMaker

El artefacto `model.tar.gz` debe contener:
- `model.joblib` — el pipeline serializado
- `inference.py` — lógica de entrada/salida que SageMaker ejecuta en el container

`inference.py` debe redefinir `FeatureEngineer` con el mismo nombre de clase para que `joblib` pueda deserializarla. El container de sklearn configura `PYTHONPATH` al directorio donde vive `inference.py`, por lo que la clase queda importable.

In [ ]:
joblib.dump(calibrated, 'model.joblib')
print(f"model.joblib  {Path('model.joblib').stat().st_size / 1e6:.1f} MB")

In [ ]:
%%writefile inference.py
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

INPUT_FEATURES = ['air_temp', 'process_temp', 'speed', 'torque', 'tool_wear', 'machine_type']


class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=INPUT_FEATURES)
        X['power']     = X['torque'] * (X['speed'] * 2 * np.pi / 60)
        X['temp_diff'] = X['process_temp'] - X['air_temp']
        X['strain']    = X['torque'] * X['tool_wear']
        return X


def model_fn(model_dir):
    return joblib.load(f"{model_dir}/model.joblib")


def input_fn(request_body, content_type='application/json'):
    data = json.loads(request_body)
    if isinstance(data, dict):
        data = [data]
    return pd.DataFrame(data)[INPUT_FEATURES]


def predict_fn(input_data, model):
    probs = model.predict_proba(input_data)[:, 1]
    preds = (probs >= 0.5).astype(int)
    return [
        {'failure_prob': round(float(p), 4), 'failure': int(f)}
        for p, f in zip(probs, preds)
    ]


def output_fn(prediction, accept='application/json'):
    body = prediction[0] if len(prediction) == 1 else prediction
    return json.dumps(body), 'application/json'

In [ ]:
with tarfile.open('model.tar.gz', 'w:gz') as tar:
    tar.add('model.joblib')
    tar.add('inference.py')

members = [m.name for m in tarfile.open('model.tar.gz').getmembers()]
print(f"model.tar.gz — contenido: {members}")
print(f"Tamaño: {Path('model.tar.gz').stat().st_size / 1e6:.1f} MB")

## 12. Despliegue en SageMaker

### Container

| Container | Cuándo usarlo |
|---|---|
| `sagemaker-scikit-learn:1.2-1-cpu-py3` | **Este modelo** — sklearn Pipeline con LR / RF / HGBC |
| `sagemaker-xgboost:1.7-1` | Solo modelos XGBoost nativos (`xgb.train` / `xgb.Booster`); no acepta sklearn Pipeline |
| Imagen ECR propia | Cuando se necesita xgboost dentro de un sklearn Pipeline, o dependencias no disponibles en los containers administrados |

### Pasos de despliegue

```python
import boto3, sagemaker
from sagemaker.sklearn import SKLearnModel

# 1. Subir artefacto a S3
s3_uri = sagemaker.Session().upload_data('model.tar.gz', key_prefix='predictive-maintenance')

# 2. Definir modelo
model = SKLearnModel(
    model_data=s3_uri,
    role='<ARN del rol de ejecución>',
    framework_version='1.2-1',
    py_version='py3',
    entry_point='inference.py'   # ya está dentro del tar.gz
)

# 3. Desplegar endpoint
predictor = model.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large'
)
```

### Schema de entrada / salida

**Request** (`application/json`):
```json
{"air_temp": 300.0, "process_temp": 310.0, "speed": 1500, "torque": 40.0, "tool_wear": 100, "machine_type": "M"}
```

**Response** (`application/json`):
```json
{"failure_prob": 0.0312, "failure": 0}
```

El umbral de decisión (`>= 0.5`) es configurable en `predict_fn`. Con el modelo calibrado, `failure_prob` tiene interpretación directa: un valor de 0.8 significa que en el 80% de los casos con esa predicción el fallo es real.